> `oeai_logger.ipynb`
> [20251105.1]
> *OEAI Logger class*

In [0]:
from types import SimpleNamespace
import builtins as _lib_builtins
import logging as _lib_logging
import json as _lib_json
import uuid as _lib_uuid
import time as _lib_time
import threading as _lib_threading
import io as _lib_io
import os as _lib_os
from typing import Any, Dict, Iterable, Optional, Tuple
import datetime as _lib_datetime
import urllib.parse as _lib_urllib_parse
import pyspark.sql.functions as _lib_pyspark_sql_functions
import pprint as _lib_pprint  # used by __repr__

In [0]:
# DBTITLE 1,class StreamFormatter
# -------- Stream Formatter (icon + exception-aware) ---------------------------
class StreamFormatter(_lib_logging.Formatter):
    """
    Human-friendly console formatter:
      - Icon + [LEVEL] prefix
      - Timestamp with milliseconds
      - Full traceback when exc_info is present (e.g., via .exception())
    """
    ICON = {
        _lib_logging.NOTSET:   "🟤",
        _lib_logging.DEBUG:    "➖",
        _lib_logging.INFO:     "ℹ️",
        _lib_logging.WARNING:  "⚠️",
        _lib_logging.ERROR:    "🔶",
        _lib_logging.CRITICAL: "❌",
    }

    def __init__(self, *, datefmt: str = "%Y-%m-%d %H:%M:%S"):
        super().__init__(datefmt=datefmt)

    def format(self, record: _lib_logging.LogRecord) -> str:
        ts = self.formatTime(record, self.datefmt)
        ts = f"{ts}.{int(record.msecs):03d}"

        icon = self.ICON.get(getattr(record, "levelno", _lib_logging.NOTSET), "🟤")
        level = record.levelname
        msg = record.getMessage()

        out = f"{icon} {ts} [{level:<8}]\t{msg}"

        if record.exc_info:
            try:
                exc_type = record.exc_info[0].__name__
                exc_msg  = str(record.exc_info[1])
            except Exception:
                exc_type, exc_msg = "Exception", ""
            out += f"  —  {exc_type}: {exc_msg}"
            try:
                out += "\n" + self.formatException(record.exc_info)
            except Exception:
                out += "\n<exception formatting failed>"

        return out


In [0]:
# DBTITLE 1,class JSONFormatter
# -------- JSON Formatter (structured logs) ------------------------------------
class JSONFormatter(_lib_logging.Formatter):
    """
    Compact JSON output with explicit separation of:
      - std:   standard context (client_id, platform, notebook_name, run_id, pipeline_id, ...)
      - event: per-call additional attributes (ad-hoc, non-standard)
      - block: block-scoped attributes (block_event, block_id, block_parent_id, block_type, block_name)
      - exception: when present, a dict with type/message/traceback
    Emits top-level: timestamp, level, message, event_id, event_number, std, event, block, exception?
    """
    def formatTime(self, record, datefmt=None):
        dt = _lib_datetime.datetime.utcfromtimestamp(record.created)
        s = dt.isoformat(timespec="milliseconds")
        return s + "Z"

    def format(self, record: _lib_logging.LogRecord) -> str:
        data: Dict[str, Any] = {
            "timestamp": self.formatTime(record),
            "level": record.levelname,
            "message": record.getMessage(),
        }
        if hasattr(record, "event_id"):
            data["event_id"] = str(record.event_id)
        if hasattr(record, "event_number"):
            data["event_number"] = record.event_number

        std = getattr(record, "std", {}) or {}
        event = getattr(record, "event", {}) or {}
        block = getattr(record, "block", {}) or {}

        std = dict(std)

        data["std"] = std
        data["event"] = event
        data["block"] = block

        # Bundle exceptions into a single object
        if record.exc_info:
            try:
                exc = {
                    "type": record.exc_info[0].__name__,
                    "message": str(record.exc_info[1]),
                    "traceback": self.formatException(record.exc_info),
                }
                data["exception"] = exc
            except Exception:
                pass

        # return _lib_json.dumps(data, separators=(",", ":"))

        def _json_default(obj):
            import datetime as _lib_dt
            import decimal as _lib_decimal
            import uuid as _lib_uuid
            if isinstance(obj, (_lib_dt.datetime, _lib_dt.date)):
                return obj.isoformat()
            if isinstance(obj, _lib_decimal.Decimal):
                # choose float or str depending on your needs
                return float(obj)
            if isinstance(obj, _lib_uuid.UUID):
                return str(obj)
            if isinstance(obj, set):
                return list(obj)
            if isinstance(obj, bytes):
                try:
                    return obj.decode("utf-8", errors="replace")
                except Exception:
                    return obj.hex()
            # Last-ditch fallback
            try:
                return str(obj)
            except Exception:
                return repr(obj)

        return _lib_json.dumps(data, separators=(",", ":"), default=_json_default)        

In [0]:
# DBTITLE 1,class FSSpecFileHandler
# -------- fsspec FileHandler with optional rotation ---------------------------
class FSSpecFileHandler(_lib_logging.Handler):
    """
    File-like logging handler that writes to fsspec URLs and supports rotation.
    """
    def __init__(
        self,
        path: str,
        *,
        oeai: Any = None,
        storage_options: Optional[Dict[str, Any]] = None,
        encoding: str = "utf-8",
        rotation_method: str = "none",   # 'none' | 'size' | 'time'
        max_bytes: int = 10 * 1024 * 1024,
        when: str = "D",
        interval: int = 1,
        backup_count: int = 7,
    ):
        super().__init__()
        self.path = path
        self.oeai = oeai
        self.storage_options = dict(storage_options or {})
        self.encoding = encoding

        self.rotation_method = rotation_method.lower()
        self.max_bytes = int(max_bytes)
        self.when = when.upper()
        self.interval = int(interval)
        self.backup_count = int(backup_count)

        self._fs = None
        self._fo = None
        self._normalized_url = None
        self._rollover_at_ts: Optional[float] = None

        self._ensure_fs_and_open(append=True)

    def emit(self, record: _lib_logging.LogRecord) -> None:
        try:
            s = self.format(record)
        except Exception:
            self.handleError(record)
            return

        try:
            if self._needs_rollover():
                self._do_rollover()
            self._fo.write(s + "\n")
            self._fo.flush()
        except Exception:
            self.handleError(record)

    def flush(self) -> None:
        try:
            if self._fo:
                self._fo.flush()
        except Exception:
            pass

    def close(self) -> None:
        try:
            if self._fo:
                self._fo.flush()
                self._fo.close()
        finally:
            self._fo = None
            super().close()

    def _ensure_fs_and_open(self, append: bool = True) -> None:
        if self._fs is None:
            self._fs, self._normalized_url = self._ensure_fs(self.path)

        try:
            parsed = _lib_urllib_parse.urlparse(self._normalized_url)
            if parsed.scheme in ("", "file"):
                parent = _lib_os.path.dirname(parsed.path)
                if parent and not _lib_os.path.exists(parent):
                    _lib_os.makedirs(parent, exist_ok=True)
        except Exception:
            pass

        mode = "a" if append else "w"
        import fsspec as _lib_fsspec
        self._fo = _lib_fsspec.open(
            self._normalized_url, mode=mode, encoding=self.encoding,
            **(self.storage_options or {})
        ).open()

        if self.rotation_method == "time":
            now = _lib_time.time()
            self._rollover_at_ts = self._compute_next_rollover(now)

    def _ensure_fs(self, url: str) -> Tuple[Any, str]:
        import fsspec as _lib_fsspec
        parsed = _lib_urllib_parse.urlparse(url)
        proto = (parsed.scheme or "").lower()

        if proto in ("abfss","abfs"):
            acct = parsed.netloc.split("@")[-1].split(".")[0] if "@" in parsed.netloc else parsed.netloc.split(".")[0]
            _container = parsed.username if parsed.username else parsed.netloc.split("@")[0]

            if not self.storage_options.get("account_name"):
                self.storage_options["account_name"] = acct

            if not any(k in self.storage_options for k in ("account_key","credential")) and self.oeai is not None:
                try:
                    conf_key = f"fs.azure.account.key.{acct}.dfs.core.windows.net"
                    account_key = self.oeai.spark.conf.get(conf_key)
                    if account_key:
                        self.storage_options["account_key"] = account_key
                except Exception:
                    pass

            fs = _lib_fsspec.filesystem(proto, **self.storage_options)
            norm_url = url
            return fs, norm_url

        fs = _lib_fsspec.filesystem(proto or "file", **self.storage_options)
        norm_url = url
        return fs, norm_url

    def _needs_rollover(self) -> bool:
        try:
            if self.rotation_method == "size":
                try:
                    size = self._fs.size(self._normalized_url)
                except Exception:
                    size = getattr(self._fo, "size", None)
                return (size or 0) >= self.max_bytes

            if self.rotation_method == "time":
                return self._rollover_at_ts is not None and _lib_time.time() >= self._rollover_at_ts

            return False
        except Exception:
            return False

    def _do_rollover(self) -> None:
        try:
            self._fo.flush()
            self._fo.close()
        except Exception:
            pass

        try:
            for i in _lib_builtins.range(self.backup_count - 1, 0, -1):
                src = f"{self._normalized_url}.{i}"
                dst = f"{self._normalized_url}.{i+1}"
                if self._fs.exists(src):
                    if self._fs.exists(dst):
                        self._fs.rm(dst)
                    self._fs.rename(src, dst)
            if self._fs.exists(self._normalized_url):
                dst = f"{self._normalized_url}.1"
                if self._fs.exists(dst):
                    self._fs.rm(dst)
                self._fs.rename(self._normalized_url, dst)
        except Exception as e:
            _lib_builtins.print(f"[oeai.log] Rotation warning: {e}")

        self._ensure_fs_and_open(append=False)
        if self.rotation_method == "time":
            self._rollover_at_ts = self._compute_next_rollover(_lib_time.time())

    def _compute_next_rollover(self, now_ts: float) -> float:
        when = self.when
        if when == "S":
            delta = _lib_datetime.timedelta(seconds=self.interval)
        elif when == "M":
            delta = _lib_datetime.timedelta(minutes=self.interval)
        elif when == "H":
            delta = _lib_datetime.timedelta(hours=self.interval)
        elif when == "D":
            delta = _lib_datetime.timedelta(days=self.interval)
        else:
            delta = _lib_datetime.timedelta(days=1)
        return (_lib_datetime.datetime.utcfromtimestamp(now_ts) + delta).timestamp()

In [0]:
# -------- Batching HTTP Handler (async) ---------------------------------------
class BatchingHTTPHandler(_lib_logging.Handler):
    """
    Asynchronous HTTP handler with batching and retries.
    """
    def __init__(
        self,
        api_base_url: str,  # renamed from api_url (required)
        api_key: Optional[str] = None,
        *,
        api_service_path: str = "logger/api",   # new: defaults but overridable
        default_api_route: str = "batch",       # new: internal default route
        flush_batch_size: int = 20,
        flush_interval_s: float = 10.0,
        max_buffer_bytes: int = 256 * 1024,
        urgent_event_level: int = _lib_logging.WARNING,
        urgent_event_types: Optional[Iterable[str]] = None,
        flush_on_urgent: bool = True,
        extra_headers: Optional[Dict[str, str]] = None,
        timeout_s: float = 8.0,
        verify_tls: bool = True,
        background_flush: bool = True,
        print_response: bool = False,
        response_max_chars: int = 500,
        queue_maxsize: int = 10000,
        enqueue_timeout_s: float = 0.05,
        urgent_synchronous: bool = False,
        max_retries: int = 3,
        backoff_base_s: float = 0.5,
        backoff_factor: float = 2.0,
        retry_jitter_s: float = 0.2,
        retry_statuses: Optional[Iterable[int]] = None,
        respect_retry_after: bool = True,
    ):
        super().__init__()
        # --- begin requested changes ---
        self.api_base_url = api_base_url
        self.api_service_path = api_service_path
        self.api_route = default_api_route
        # --- end requested changes ---
        self.api_key = api_key
        self.flush_batch_size = int(flush_batch_size)
        self.flush_interval_s = float(flush_interval_s)
        self.max_buffer_bytes = int(max_buffer_bytes)

        self.urgent_event_level = int(urgent_event_level)
        self.urgent_event_types = set(urgent_event_types or ())
        self.flush_on_urgent = bool(flush_on_urgent)

        self.timeout_s = float(timeout_s)
        self.verify_tls = bool(verify_tls)
        self.extra_headers = dict(extra_headers or {})

        self.print_response = bool(print_response)
        self.response_max_chars = int(response_max_chars)

        import queue as _lib_queue
        self._q: "_lib_queue.Queue[tuple[str,bool]]" = _lib_queue.Queue(maxsize=queue_maxsize)
        self.enqueue_timeout_s = float(enqueue_timeout_s)
        self.urgent_synchronous = bool(urgent_synchronous)

        self.max_retries = int(max_retries)
        self.backoff_base_s = float(backoff_base_s)
        self.backoff_factor = float(backoff_factor)
        self.retry_jitter_s = float(retry_jitter_s)
        self.retry_statuses = set(retry_statuses or {429, 500, 502, 503, 504})
        self.respect_retry_after = bool(respect_retry_after)

        self._pending: list[str] = []
        self._pending_bytes = 0
        self._last_flush = _lib_time.time()

        self._stop_evt = _lib_threading.Event()
        self._flush_now_evt = _lib_threading.Event()
        self._flush_ack_evt = _lib_threading.Event()
        self._worker: Optional[_lib_threading.Thread] = None

        if background_flush and self.flush_interval_s > 0:
            self._worker = _lib_threading.Thread(target=self._sender_loop, name="oeai_http_sender", daemon=True)
            self._worker.start()

    def emit(self, record: _lib_logging.LogRecord) -> None:
        try:
            s = self.format(record)
        except Exception:
            self.handleError(record)
            return

        level = getattr(record, "levelno", _lib_logging.INFO)

        # check block_event for urgency
        ev_type = None
        try:
            blk = getattr(record, "block", None)
            if isinstance(blk, dict):
                ev_type = blk.get("block_event")
            if ev_type is None:
                ev_type = getattr(record, "block_event", None)
        except Exception:
            ev_type = None

        urgent = (level >= self.urgent_event_level) or (ev_type in self.urgent_event_types)

        if urgent and self.urgent_synchronous:
            if self.flush_on_urgent and self._pending:
                batch = self._pending + [s]
                self._send_batch(batch)
                self._pending.clear()
                self._pending_bytes = 0
            else:
                self._send_batch([s])
            self._last_flush = _lib_time.time()
            return

        try:
            self._q.put((s, urgent), timeout=self.enqueue_timeout_s)
        except Exception:
            try:
                _ = self._q.get_nowait()
                self._q.put_nowait((s, urgent))
            except Exception:
                _lib_builtins.print("[oeai.log] HTTP queue full; dropping log event")

        if urgent:
            self._flush_now_evt.set()

    def flush(self) -> None:
        if self._worker and self._worker.is_alive():
            self._flush_ack_evt.clear()
            self._flush_now_evt.set()
            max_wait = self.timeout_s * (self.max_retries + 1) + 5.0
            self._flush_ack_evt.wait(timeout=max_wait)
        else:
            if self._pending:
                self._send_batch(self._pending)
                self._pending.clear()
                self._pending_bytes = 0
                self._last_flush = _lib_time.time()

    def close(self) -> None:
        try:
            self._stop_evt.set()
            self._flush_now_evt.set()
            if self._worker and self._worker.is_alive():
                self._worker.join(timeout=2.0 * (self.timeout_s * (self.max_retries + 1) + 5.0))
        finally:
            super().close()

    def _sender_loop(self):
        def thresholds() -> bool:
            return (_lib_builtins.len(self._pending) >= self.flush_batch_size) or (self._pending_bytes >= self.max_buffer_bytes)

        while not self._stop_evt.is_set():
            now = _lib_time.time()
            timeout = _lib_builtins.max(0.25, self.flush_interval_s - (now - self._last_flush))
            if self._flush_now_evt.is_set():
                timeout = 0.01

            try:
                js, urgent = self._q.get(timeout=timeout)
                self._pending.append(js)
                self._pending_bytes += _lib_builtins.len(js) + 1

                if urgent and self.flush_on_urgent:
                    self._send_batch(self._pending)
                    self._pending.clear()
                    self._pending_bytes = 0
                    self._last_flush = _lib_time.time()
                elif urgent and not self.flush_on_urgent:
                    self._send_batch([js])
                    self._last_flush = _lib_time.time()
                elif thresholds():
                    self._send_batch(self._pending)
                    self._pending.clear()
                    self._pending_bytes = 0
                    self._last_flush = _lib_time.time()

            except Exception:
                elapsed = _lib_time.time() - self._last_flush
                if self._pending and (elapsed >= self.flush_interval_s or self._flush_now_evt.is_set()):
                    self._send_batch(self._pending)
                    self._pending.clear()
                    self._pending_bytes = 0
                    self._last_flush = _lib_time.time()
                if self._flush_now_evt.is_set():
                    self._flush_now_evt.clear()
                    self._flush_ack_evt.set()

        # drain on stop
        while True:
            try:
                js, _ = self._q.get_nowait()
                self._pending.append(js)
                self._pending_bytes += _lib_builtins.len(js) + 1
            except Exception:
                break
        if self._pending:
            self._send_batch(self._pending)
            self._pending.clear()
            self._pending_bytes = 0
            self._last_flush = _lib_time.time()
        self._flush_ack_evt.set()

    def _sleep_with_jitter(self, base: float):
        try:
            import random as _lib_random
            _lib_time.sleep(_lib_builtins.max(0.0, base + _lib_random.random() * self.retry_jitter_s))
        except Exception:
            _lib_time.sleep(base)

    def _parse_retry_after(self, hdr: Optional[str]) -> Optional[float]:
        if not hdr:
            return None
        try:
            hdr = hdr.strip()
            if hdr.isdigit():
                return float(hdr)
            import email.utils as _lib_email_utils
            dt = _lib_email_utils.parsedate_to_datetime(hdr)
            return _lib_builtins.max(0.0, (dt - _lib_datetime.datetime.utcnow()).total_seconds())
        except Exception:
            return None

    # --- begin requested addition ---
    def _endpoint_url(
        self,
        route: Optional[str] = None,
        service_path: Optional[str] = None,
        base_url: Optional[str] = None,
    ) -> str:
        from urllib.parse import urljoin
        base = (base_url or self.api_base_url).rstrip('/') + '/'
        svc  = (self.api_service_path if service_path is None else service_path).strip('/')
        r    = (self.api_route      if route        is None else route).strip('/')
        path = '/'.join(p for p in (svc, r) if p)
        return urljoin(base, path)
    # --- end requested addition ---

    def _send_batch(self, batch: list[str]) -> None:
        if not batch:
            return
        try:
            try:
                import requests as _lib_requests
            except Exception as e:
                _lib_builtins.print(f"[oeai.log] HTTP disabled (requests import failed): {e}")
                return

            payload = "[" + ",".join(batch) + "]"
            headers = {"Content-Type": "application/json"}
            headers.update(self.extra_headers)
            if self.api_key:
                headers.setdefault("Ocp-Apim-Subscription-Key", self.api_key)
                # headers.setdefault("Authorization", f"Bearer {self.api_key}")

            attempt = 0
            while True:
                attempt += 1
                try:
                    # --- begin requested change: build URL from parts ---
                    url = self._endpoint_url()
                    resp = _lib_requests.post(
                        url, data=payload, headers=headers,
                        timeout=self.timeout_s, verify=self.verify_tls
                    )
                    # --- end requested change ---

                    will_retry = resp.status_code in self.retry_statuses and attempt <= self.max_retries
                    if will_retry:
                        delay = self.backoff_base_s * (self.backoff_factor ** (attempt - 1))
                        if self.respect_retry_after:
                            ra = self._parse_retry_after(resp.headers.get("Retry-After"))
                            if ra is not None:
                                delay = _lib_builtins.max(delay, float(ra))
                        self._sleep_with_jitter(delay)
                        continue

                    if resp.status_code >= 400:
                        _lib_builtins.print(f"[oeai.log] HTTP {resp.status_code}: {resp.text[:300]}")
                    break

                except (_lib_requests.exceptions.ReadTimeout,
                        _lib_requests.exceptions.ConnectTimeout,
                        _lib_requests.exceptions.ConnectionError) as e:
                    if attempt <= self.max_retries:
                        delay = self.backoff_base_s * (self.backoff_factor ** (attempt - 1))
                        self._sleep_with_jitter(delay)
                        continue
                    _lib_builtins.print(f"[oeai.log] HTTP send failed: {e}")
                    break

                except Exception as e:
                    _lib_builtins.print(f"[oeai.log] HTTP send failed: {e}")
                    break
        except Exception as e:
            _lib_builtins.print(f"[oeai.log] HTTP send failed: {e}")


In [0]:
# DBTITLE 1,class OEAILogger
# -------- OEAILogger (public facade) ------------------------------------------
class OEAILogger:
    """
    Structured logger with std/event/block sections; deterministic event_id; block stack helpers.
    """

    RESERVED_LOGRECORD_KEYS = {
        "name","msg","args","levelname","levelno","pathname","filename",
        "module","exc_info","exc_text","stack_info","lineno","funcName",
        "created","msecs","relativeCreated","thread","threadName","process",
        "processName","message","asctime"
    }

    def __init__(self, *, namespace_uuid: Optional[_lib_uuid.UUID] = None):
        self.logger: _lib_logging.Logger = _lib_logging.getLogger("oeai")
        self.logger.setLevel(_lib_logging.DEBUG)
        self.logger.propagate = False
        if not any(isinstance(h, _lib_logging.NullHandler) for h in self.logger.handlers):
            self.logger.addHandler(_lib_logging.NullHandler())

        self.block_id_list: list[str] = []
        self.checkpoints = [_lib_time.perf_counter()]
        self.handler = SimpleNamespace()
        self.extra_std: Dict[str, Any] = {}
        self.uuid_fingerprint: str = ""

        # Namespace UUID (moved inside class; configurable)
        self.namespace_uuid = namespace_uuid or _lib_uuid.uuid5(_lib_uuid.NAMESPACE_DNS, "oeai.logging")

        # Formatters (reverted to simple stream formatter)
        self.formatter = SimpleNamespace()
        self.formatter.default = _lib_logging.Formatter(
            fmt="%(asctime)s.%(msecs)03d [%(levelname)s] %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S",
        )
        self.formatter.stream = StreamFormatter()
        self.formatter.json = JSONFormatter()

    def __repr__(self):
        return _lib_pprint.pformat(self.__dict__, indent=2)

    # Alias __str__ to __repr__ so print(o) uses the same
    __str__ = __repr__

    # ---- run/pipeline id resolution (moved into class) -----------------------
    def _get_run_and_pipeline_ids(self, spark):
        """
        Returns (run_id, pipeline_id) derived from platform context with timestamped fallbacks.
        Uses platform from self.extra_std (if available) or oeai.platform.
        """
        plat = ((getattr(self, "extra_std", None) or {}).get("platform")
                or getattr(oeai, "platform", None) or "").lower()
        run_id = None
        pipeline_id = None

        notebook_name = (
            getattr(getattr(oeai, "notebook", None), "name", None)
            or getattr(oeai, "notebook_name", None)
            or getattr(oeai, "notebook_id", None)
            or "notebook"
        )

        if plat == "databricks":
            try:
                import pyspark.dbutils as _lib_pyspark_dbutils
                dbutils = _lib_pyspark_dbutils.DBUtils(spark)
                tags = dbutils.notebook().getContext().tags()
                run_id = tags.get("jobRunId") or tags.get("runId")
                pipeline_id = tags.get("jobId") or spark.conf.get("pipeline_id", None)
            except Exception:
                run_id = None
                try:
                    pipeline_id = spark.conf.get("pipeline_id", None)
                except Exception:
                    pipeline_id = None

            if not run_id:
                run_id = f"{notebook_name}_{int(_lib_time.time())}"

        elif plat == "fabric":
            try:
                pipeline_id = spark.conf.get("pipeline_id", None)
            except Exception:
                pipeline_id = None
            run_id = f"{notebook_name}_{int(_lib_time.time())}"

        else:
            cid = getattr(oeai, "client_id", "client")
            run_id = f"{cid}_{notebook_name}_{int(_lib_time.time())}"
            pipeline_id = None

        return run_id, pipeline_id

    # -- internals
    def _build_extra(self) -> Dict[str, Any]:
        """
        Build standard context once per run and compute a stable fingerprint.
        """
        std = {
            "client_id": getattr(oeai, "client_id", None),
            "platform": getattr(oeai, "platform", None),
            "module_id": getattr(oeai, "module_id", None),
        }

        if hasattr(oeai, "notebook"):
            std["notebook_name"] = getattr(oeai.notebook, "name", None)
            std["notebook_buildversion"] = getattr(oeai.notebook, "buildversion", None)
            std["notebook_buildtimestamp"] = getattr(oeai.notebook, "buildtimestamp", None)

        try:
            import pyspark.sql as _lib_pyspark_sql
            spark = _lib_pyspark_sql.SparkSession.getActiveSession() or _lib_pyspark_sql.SparkSession.getOrCreate()
        except Exception:
            spark = None
        run_id, pipeline_id = self._get_run_and_pipeline_ids(spark)

        std["run_id"] = run_id
        if pipeline_id is not None:
            std["pipeline_id"] = pipeline_id

        std = {k: v for k, v in std.items() if v is not None}

        fp = _lib_json.dumps({k: std[k] for k in sorted(std.keys())},
                             sort_keys=True, separators=(",", ":"))
        self.uuid_fingerprint = fp

        self.extra_std = std
        return std

    @staticmethod
    def _split_logging_kwargs(kwargs: Dict[str, Any]) -> Tuple[Dict[str, Any], Dict[str, Any]]:
        """
        Separate logging control kwargs (exc_info, stack_info, stacklevel, extra)
        from caller-supplied attributes (attrs).
        """
        known = {"exc_info", "stack_info", "stacklevel", "extra", "event"}  # allow 'event' convenience key
        log_kwargs = {k: v for k, v in kwargs.items() if k in known}
        attrs = {k: v for k, v in kwargs.items() if k not in known}
        return log_kwargs, attrs

    # -- public setup
    def init(self) -> None:
        """Attach handlers using configured properties. Safe to call repeatedly."""
        self.extra_std = self._build_extra()
        self._event_seq = 0

        self.logger.handlers = [h for h in self.logger.handlers if isinstance(h, _lib_logging.NullHandler)]

        if hasattr(self.handler, "stream"):
            try:
                sh = _lib_logging.StreamHandler()
                sh.setLevel(getattr(self.handler.stream, "level", _lib_logging.INFO))
                fmt_name = getattr(self.handler.stream, "formatter", "stream")
                sh.setFormatter(getattr(self.formatter, fmt_name, self.formatter.stream))
                self.logger.addHandler(sh)
            except Exception as e:
                _lib_builtins.print(f"[oeai.log] Failed to add StreamHandler: {e}")

        if hasattr(self.handler, "adls") and getattr(self.handler.adls, "path", None):
            try:
                fh = FSSpecFileHandler(
                    self.handler.adls.path,
                    oeai=oeai,
                    storage_options=(getattr(self.handler.adls, "storage_options", None) or {}),
                    rotation_method=getattr(self.handler.adls, "rotation_method", "none"),
                    max_bytes=getattr(self.handler.adls, "max_bytes", 10 * 1024 * 1024),
                    when=getattr(self.handler.adls, "when", "D"),
                    interval=getattr(self.handler.adls, "interval", 1),
                    backup_count=getattr(self.handler.adls, "backup_count", 7),
                )
                fh.setLevel(getattr(self.handler.adls, "level", _lib_logging.DEBUG))
                fmt_name = getattr(self.handler.adls, "formatter", "json")
                fh.setFormatter(getattr(self.formatter, fmt_name, self.formatter.json))
                self.logger.addHandler(fh)
            except Exception as e:
                _lib_builtins.print(f"[oeai.log] Failed to add ADLS FileHandler: {e}")

        if hasattr(self.handler, "http") and getattr(self.handler.http, "api_base_url", None):
            try:
                cfg = self.handler.http
                cfg_dict = vars(cfg).copy() if hasattr(cfg, "__dict__") else dict(cfg)

                api_base_url = cfg_dict.pop("api_base_url")
                api_key = cfg_dict.pop("api_key", None)
                level   = cfg_dict.pop("level", _lib_logging.INFO)
                fmt_name= cfg_dict.pop("formatter", "json")

                import inspect as _lib_inspect
                sig = _lib_inspect.signature(BatchingHTTPHandler.__init__)
                allowed_keys = set(sig.parameters) - {"self", "api_base_url", "api_key"}
                kwargs = {k: v for k, v in cfg_dict.items() if k in allowed_keys}

                http = BatchingHTTPHandler(api_base_url, api_key=api_key, **kwargs)
                http.setLevel(level)
                http.setFormatter(getattr(self.formatter, fmt_name, self.formatter.json))
                self.logger.addHandler(http)
            except Exception as e:
                _lib_builtins.print(f"[oeai.log] Failed to add HTTP handler: {e}")

        self._log(_lib_logging.INFO, "oeai.log initialized", block_event="log.init")

    def shutdown(self) -> None:
        # Emit a shutdown event so downstream aggregations can mark the run as finished.
        self._log(
            _lib_logging.INFO,
            "oeai.log shutdown",
            event={"event_type": "log.shutdown"}
        )

        handlers = [h for h in self.logger.handlers if not isinstance(h, _lib_logging.NullHandler)]
        for h in handlers:
            try: h.flush()
            except Exception: pass
            try: h.close()
            except Exception: pass
        self.logger.handlers = [h for h in self.logger.handlers if isinstance(h, _lib_logging.NullHandler)]

    # -- level wrappers ----------------------------------------------------------
    def notset(self, msg: str, *args, **kwargs):   self._log(_lib_logging.NOTSET, msg, *args, **kwargs)
    def debug(self, msg: str, *args, **kwargs):    self._log(_lib_logging.DEBUG, msg, *args, **kwargs)
    def info(self, msg: str, *args, **kwargs):     self._log(_lib_logging.INFO, msg, *args, **kwargs)
    def warning(self, msg: str, *args, **kwargs):  self._log(_lib_logging.WARNING, msg, *args, **kwargs)
    def error(self, msg: str, *args, **kwargs):    self._log(_lib_logging.ERROR, msg, *args, **kwargs)
    def critical(self, msg: str, *args, **kwargs): self._log(_lib_logging.CRITICAL, msg, *args, **kwargs)
    def exception(self, msg: str, *args, **kwargs):
        kwargs["exc_info"] = True
        self._log(_lib_logging.ERROR, msg, *args, **kwargs)

    def _log(self, level: int, msg: str, *args, **kwargs) -> None:
        """
        Central logging path. Builds std/event/block in-place.
        - std:   from self.extra_std
        - block: block_* keys (incl. legacy remaps)
        - event: remaining ad-hoc keys; 'event' kwarg can supply a dict explicitly
        """
        # Split logging-control kwargs vs caller attrs
        log_kwargs, attrs = self._split_logging_kwargs(kwargs)

        # Prepare std
        std = self.extra_std or self._build_extra()
        self.extra_std = std

        # Sequence and deterministic event_id
        if not hasattr(self, "_event_seq"):
            self._event_seq = 0
        self._event_seq += 1
        event_number = self._event_seq
        fingerprint = self.uuid_fingerprint or ""
        event_id = str(_lib_uuid.uuid5(self.namespace_uuid, f"{fingerprint}:{event_number}"))

        # Build block and event dicts
        block: Dict[str, Any] = {}
        event: Dict[str, Any] = {}

        # Allow explicit 'event={...}' dict for convenience
        if "event" in log_kwargs and isinstance(log_kwargs["event"], dict):
            event.update(log_kwargs["event"])
            log_kwargs.pop("event", None)

        # If user passed 'extra' (classic logging), treat it as ad-hoc event fields
        if "extra" in log_kwargs and isinstance(log_kwargs["extra"], dict):
            event.update(log_kwargs["extra"])

        # Merge remaining attrs into event (ad-hoc), but extract block-related ones
        # 1) legacy → new
        if "event_type" in attrs:
            attrs["block_event"] = attrs.pop("event_type")
        if "parent_block_id" in attrs:
            attrs["block_parent_id"] = attrs.pop("parent_block_id")

        # 2) pick block_* keys
        for k in ("block_event", "block_id", "block_parent_id", "block_type", "block_name"):
            if k in attrs:
                block[k] = attrs.pop(k)

        # 3) whatever remains goes into event (with reserved key guard)
        for k, v in attrs.items():
            if k in self.RESERVED_LOGRECORD_KEYS:
                event[f"oeai_{k}"] = v
            else:
                event[k] = v

        # Ensure current block_id is always present
        if "block_id" not in block:
            block["block_id"] = self.block_id_list[-1] if self.block_id_list else None

        # Attach structured extras
        log_kwargs["extra"] = {
            "std": std,
            "event": event,
            "block": block,
            "event_id": event_id,
            "event_number": event_number
        }

        self.logger.log(level, msg, *args, **log_kwargs)

    # -- blocks (name first; type optional) ------------------------------------
    def start_block(self, block_name: str, block_type: Optional[str] = None, *, block_id: Optional[str] = None) -> str:
        """
        Push a block onto the stack and emit a block.start event (direct via _log()).
        """
        bid = block_id or _lib_uuid.uuid4().hex[:16]
        parent = self.block_id_list[-1] if self.block_id_list else None
        self.block_id_list.append(bid)

        label = f"{block_type} - {block_name}" if block_type else block_name
        self._log(
            _lib_logging.INFO,
            f"Block start: {label}",
            block_event="block.start",
            block_id=bid,
            block_parent_id=parent,
            block_name=block_name,
            **({"block_type": block_type} if block_type is not None else {})
        )
        return bid

    def end_block(
        self,
        *,
        block_id: Optional[str] = None,
        index: Optional[int] = None,
        levels: Optional[int] = None,
        include_target: bool = True
    ) -> None:
        """
        Pop and close blocks; emits block.end events (direct via _log()).
        """
        choices = ((block_id is not None) + (index is not None) + (levels is not None))
        if choices > 1:
            raise ValueError("Provide only one of: block_id, index, levels")
        if not self.block_id_list:
            return

        if choices == 0:
            bid = self.block_id_list.pop()
            self._log(_lib_logging.INFO, "Block end", block_event="block.end", block_id=bid)
            return

        if levels is not None:
            n = _lib_builtins.max(0, _lib_builtins.min(int(levels), _lib_builtins.len(self.block_id_list)))
            for _ in _lib_builtins.range(n):
                bid = self.block_id_list.pop()
                self._log(_lib_logging.INFO, "Block end", block_event="block.end", block_id=bid)
            return

        if index is not None:
            n = _lib_builtins.len(self.block_id_list)
            i = index if index >= 0 else n + index
            if i < 0 or i >= n:
                return
            target_len = i if include_target else i + 1
            while _lib_builtins.len(self.block_id_list) > target_len:
                bid = self.block_id_list.pop()
                self._log(_lib_logging.INFO, "Block end", block_event="block.end", block_id=bid)
            return

        assert block_id is not None
        closed = False
        while self.block_id_list:
            bid = self.block_id_list.pop()
            self._log(_lib_logging.INFO, "Block end", block_event="block.end", block_id=bid)
            if bid == block_id:
                closed = True
                break
        if not closed:
            self._log(_lib_logging.INFO, "Orphan block end", block_event="block.end", block_id=block_id)

    def checkpoint(self, label=None):
        # now = _lib_time.perf_counter()
        # self.checkpoints.append(now)
        # if _lib_builtins.len(self.checkpoints) > 1:
        #     delta = self.checkpoints[-1] - self.checkpoints[-2]
        #     total = self.checkpoints[-1] - self.checkpoints[0]
        # msg = f"{delta:.6f} s since last, {total:.6f} s total"
        # if label:
        #     msg = f"[{label}] " + msg
        # _lib_builtins.print(msg)
        pass